In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch

from utils.framework_evaluator import PATHS_SPECTRUMS, PATHS_METRICS
from experiments_assets.dataset_generator import PATHS_DATASETS
from default_configs import DEFAULT_FRAMEWORKS_CONFIGS, DEFAULT_FRAMEWORKS_REGULARIZERS, DEFAULT_METRICS_CONFIGS

from utils.framework_evaluator import spectrum_exists
from utils.assets_management_utils import generate_8char_tag, key_management



In [7]:
dataset_tag = "FLRvmQ82"
dataset_path = key_management(PATHS_DATASETS, dataset_tag, mode='load')
log_noise_variance_values = np.array(torch.load(dataset_path, weights_only=True)['metadata']['log_noise_variance_values'])

results = {}
spectrums = {}

for framework, regularizers in DEFAULT_FRAMEWORKS_REGULARIZERS.items():
    results[framework] = {}

    framework_configuration = DEFAULT_FRAMEWORKS_CONFIGS[framework].copy()

    for regularizer in regularizers:

        if framework not in spectrums:
            spectrums[framework] = {}

        if framework != "SBL":
            framework_configuration["regularization_parameter"] = regularizer

        _, _, _, spectrum_tag, _ = spectrum_exists(framework, dataset_tag, **framework_configuration) 
        spectrum_path = key_management(PATHS_SPECTRUMS, spectrum_tag, mode='load')
        
        spectrums_result = torch.load(spectrum_path, weights_only=True)['spectrums'].numpy()
        spectrums_result = np.abs(spectrums_result)

        if framework == "SBL":
            spectrums[framework] = spectrums_result
        else:
            spectrums[framework][regularizer] = spectrums_result
            results[framework][regularizer] = {}

        for metric in ['detection_rate', 'rmse', 'false_alarm_rate']:
            metric_configuration = DEFAULT_METRICS_CONFIGS.copy()
            metric_configuration["metric"] = metric
            metric_configuration["dataset_tag"] = dataset_tag

            if metric in ['rmse', 'detection_rate']:
                metric_configuration.pop("false_alarm_threshold", None)

            metric_tag = generate_8char_tag(metric_configuration | {"spectrum_tag": spectrum_tag})                
            metric_path = key_management(PATHS_METRICS, metric_tag, mode='load')
            
            average = torch.load(metric_path, weights_only=True)['average']
            
            # Store average metric safely
            if framework == "SBL":
                results[framework][metric] = average
            else:
                results[framework][regularizer][metric] = average

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(log_noise_variance_values, results['LASSO'][0.2]['detection_rate'], marker='o', label='LASSO')
plt.title("LASSO detection rate Pd with regularizer κ = 0.2")
plt.show()